# CR-1 G0 下準備（Claude API・CPU・費用の上限 $0.50）

凍結した CR-1 の計画の G0 です。**ベンチマークの問題は使わず、分かれ道も測りません。** 無害な短い文だけで、次を確かめます。

1. Haiku 4.5 と Sonnet 4.5 で、プリフィル（途中から続きを書かせる）が通るか
2. 末尾に空白のあるプリフィルが拒否されるか
3. `extra_body` での温度 0 の指定が通るか（通れば、本番の土台の道は温度 0。通らなければ温度 1。計画の定義 3）
4. 単語の区切りの振る舞い：数字・単語・記号で終わる無害な文 9 本を、それぞれ 10 回引き直し、続きが空白・数字・記号で始まる割合
5. バッチ API で 10 件を出し、結果が戻るか・`custom_id` で引き当てられるか・終わった状態が読めるか
6. SDK（`anthropic` パッケージ）の版を記録

**使い方**
1. ランタイムは **CPU** で大丈夫です。
2. Colab の「シークレット」に `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`（どちらもノートブックからのアクセスをオン）。
3. セル1 → セル2 → セル3 の順に実行。セル3はバッチの戻りを最大 40 分待ちます。間に合わなければランタイムを切って止まるので、時間をおいて **セル1 → セル3** の順に実行してください（セル2はやり直さないでね。同じバッチを待つだけで、出し直しません）。
4. 費用は全部で $0.10 くらいの見込みです（上限 $0.50 を超えそうなら止まります）。キーはどこにも表示・保存しません。

結果は Drive の `DECISION_POINTS/cr1/G0/CR1_G0.json` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head (must contain the pre-G0 lock), key (never printed), package, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, datetime

drive.mount("/content/drive")
EXPECTED_HEAD = "4dcae3f089c8b2a7218fbd13cde5317e9f1dcc4c"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/cr1/G0")
OUT.mkdir(parents=True, exist_ok=True)

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
assert (REPO / "notes" / "cr1_lock_pre_g0.md").exists(), "段2 の固定（notes/cr1_lock_pre_g0.md）がありません。G0 は回しません。"
print("exact head:", head, "| 段2 の固定あり")

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "anthropic"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録して、このノートブックからのアクセスをオンにしてね。"
client = anthropic.Anthropic(api_key=_key)
del _key
SDK_VERSION = anthropic.__version__
print("anthropic", SDK_VERSION)
H45 = "claude-haiku-4-5-20251001"
CAP = 0.50
PRICE = {"haiku": (1.0, 5.0), "sonnet": (3.0, 15.0)}     # USD per million tokens (input, output), standard rates
STATE = OUT / "CR1_G0_state.json"
ASK = [{"role": "user", "content": "Count upward by one, separated by commas."}]
def price_of(model):
    return PRICE["sonnet"] if "sonnet" in model else PRICE["haiku"]

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1 G0 下準備（API・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1 G0', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — G0 items 1, 2, 3, 4, 6 (synchronous calls; harmless text only).
R = json.loads(STATE.read_text()) if STATE.exists() else {"started_utc": datetime.datetime.utcnow().isoformat() + "Z", "spent_usd": 0.0}
R["sdk_version"] = SDK_VERSION
R["code_head"] = head

def call(model, messages, max_tokens=30, **kw):
    if R["spent_usd"] > CAP - 0.05:
        raise RuntimeError("費用の上限に近づいたので止めました")
    try:
        m = client.messages.create(model=model, max_tokens=max_tokens, messages=messages, **kw)
        pi, po = price_of(model)
        R["spent_usd"] += m.usage.input_tokens * pi / 1e6 + m.usage.output_tokens * po / 1e6
        text = "".join(b.text for b in m.content if getattr(b, "type", "") == "text")
        return {"ok": True, "stop_reason": m.stop_reason, "text": text}
    except anthropic.APIStatusError as e:
        return {"ok": False, "status": e.status_code, "error": str(e)[:200]}
    except Exception as e:
        return {"ok": False, "status": None, "error": type(e).__name__ + ": " + str(e)[:200]}

# model ids: Haiku 4.5 is fixed by the gate; Sonnet 4.5 is looked up in the model list (recorded)
ids = [m.id for m in client.models.list(limit=100).data]
S45 = sorted([i for i in ids if i.startswith("claude-sonnet-4-5")])
R["models"] = {"haiku_4_5": H45, "haiku_4_5_listed": H45 in ids, "sonnet_4_5": S45[-1] if S45 else None}
# 1. prefill on Haiku 4.5 and Sonnet 4.5
t1 = {}
for name, model in (("haiku_4_5", H45), ("sonnet_4_5", R["models"]["sonnet_4_5"])):
    if model is None:
        t1[name] = {"ok": False, "error": "model not listed"}
        continue
    r = call(model, ASK + [{"role": "assistant", "content": "1, 2, 3,"}])
    t1[name] = {"ok": r["ok"], "continues": bool(r.get("ok")) and r["text"].lstrip().startswith("4"),
                "status": r.get("status"), "error": r.get("error")}
R["t1_prefill"] = t1

# 2. a prefill ending in whitespace is rejected?
r = call(H45, ASK + [{"role": "assistant", "content": "1, 2, 3, "}])
R["t2_trailing_space_prefill"] = {"rejected": not r["ok"], "status": r.get("status"), "error": r.get("error")}

# 3. temperature 0 through extra_body (decides 定義 3)
t3 = [call(H45, ASK + [{"role": "assistant", "content": "1, 2, 3,"}], extra_body={"temperature": 0}) for _ in range(3)]
R["t3_extra_body_temperature_0"] = {"accepted": all(x["ok"] for x in t3), "statuses": [x.get("status") for x in t3],
                                    "errors": [x.get("error") for x in t3 if not x["ok"]],
                                    "three_outputs_identical": len({x.get("text") for x in t3 if x["ok"]}) == 1 if all(x["ok"] for x in t3) else None}
R["baseline_temperature"] = 0 if R["t3_extra_body_temperature_0"]["accepted"] else 1

# 4. word-boundary behaviour (9 harmless sentences x 10 resamples, default settings)
SENT = [("number", "There are 12"), ("number", "She paid 35"), ("number", "The answer is 7"),
        ("word", "He bought apples"), ("word", "The shop sells"), ("word", "We walked to the"),
        ("symbol", "The total is 4."), ("symbol", "First, we add 3 + 5 ="), ("symbol", "The list: (a)")]
ASK4 = [{"role": "user", "content": "Write one short sentence about a small shop."}]
def kind(c):
    if not c:
        return "empty"
    ch = c[0]
    return "space" if ch.isspace() else "digit" if ch.isdigit() else "letter" if ch.isalpha() else "symbol"
t4 = []
for cat, s in SENT:
    counts = {}
    for _ in range(10):
        r = call(H45, ASK4 + [{"role": "assistant", "content": s}], max_tokens=20)
        k = kind(r["text"]) if r["ok"] else "error"
        counts[k] = counts.get(k, 0) + 1
    t4.append({"ends_with": cat, "sentence": s, "first_char_kinds": counts})
R["t4_word_boundary"] = t4
R["t6_sdk_version"] = SDK_VERSION
STATE.write_text(json.dumps(R, ensure_ascii=False, indent=1))
print(json.dumps({k: R[k] for k in ("models", "t1_prefill", "t2_trailing_space_prefill", "t3_extra_body_temperature_0", "baseline_temperature")}, ensure_ascii=False, indent=1))
print("ここまでの費用（見込み）: $%.4f" % R["spent_usd"])

In [ ]:
# Cell 3 — G0 item 5: a 10-request batch (submitted once; rerun this cell later if it has not ended).
R = json.loads(STATE.read_text())
assert "t4_word_boundary" in R, "先にセル2を実行してね"
if "t5_batch_id" not in R:
    reqs = [{"custom_id": f"g0-{i}", "params": {"model": H45, "max_tokens": 20,
             "messages": ASK + [{"role": "assistant", "content": ", ".join(str(j) for j in range(1, i + 2)) + ","}]}}
            for i in range(10)]
    b = client.messages.batches.create(requests=reqs)
    R["t5_batch_id"] = b.id
    R["t5_submitted_utc"] = datetime.datetime.utcnow().isoformat() + "Z"
    STATE.write_text(json.dumps(R, ensure_ascii=False, indent=1))
    print("バッチを出しました（10 件）")
bid = R["t5_batch_id"]
deadline = time.time() + 40 * 60
while True:
    b = client.messages.batches.retrieve(bid)
    if b.processing_status == "ended" or time.time() > deadline:
        break
    time.sleep(30)
if b.processing_status != "ended":
    STATE.write_text(json.dumps(R, ensure_ascii=False, indent=1))
    print("⏳ バッチがまだ終わっていません。ランタイムを切ります。時間をおいて、セル1 → セル3 の順に実行してね（セル2はやり直さないで。同じバッチを待ちます）。")
else:
    got, kinds, cont_ok = {}, {}, 0
    for res in client.messages.batches.results(bid):
        t = res.result.type
        kinds[t] = kinds.get(t, 0) + 1
        got[res.custom_id] = t
        if t == "succeeded":
            m = res.result.message
            pi, po = price_of(H45)
            R["spent_usd"] += 0.5 * (m.usage.input_tokens * pi / 1e6 + m.usage.output_tokens * po / 1e6)
            i = int(res.custom_id.split("-")[1])
            text = "".join(c.text for c in m.content if getattr(c, "type", "") == "text")
            cont_ok += text.lstrip().startswith(str(i + 2))
    R["t5_batch"] = {"ended": True, "result_types": kinds, "all_custom_ids_back": sorted(got) == sorted(f"g0-{i}" for i in range(10)),
                     "continued_correctly": cont_ok, "request_counts": b.request_counts.model_dump() if hasattr(b.request_counts, "model_dump") else str(b.request_counts)}
    t1ok = R["t1_prefill"]["haiku_4_5"]["ok"] and R["t1_prefill"]["haiku_4_5"]["continues"]
    t5ok = R["t5_batch"]["all_custom_ids_back"] and kinds.get("succeeded", 0) == 10
    R["verdict"] = "G0_PASS" if (t1ok and t5ok) else "STOPPED_BEFORE_MAIN_RUN"
    R["finished_utc"] = datetime.datetime.utcnow().isoformat() + "Z"
    (OUT / "CR1_G0.json").write_text(json.dumps(R, ensure_ascii=False, indent=1))
    print(R["verdict"], "| 土台の道の温度:", R["baseline_temperature"], "| SDK", R["sdk_version"], "| 費用（見込み）$%.4f" % R["spent_usd"])
notify('終わりました：G0 の結果を Drive に書きました' if (OUT / 'CR1_G0.json').exists() else 'バッチ待ちで一度止めました（あとでセル3）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()